In [ ]:
!pip install esig iisignature scikit-learn pandas numpy matplotlib scipy aeon

In [ ]:
# dyadic testing
import numpy as np
import iisignature
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from aeon.datasets import load_classification
from itertools import combinations
import numba
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional


@dataclass
class Config:
    """Configuration settings for the dyadic interval feature extraction pipeline."""
    sig_level: int = 4
    max_level: int = 4
    n_estimators: int = 100
    random_state: int = 42
    k_folds: int = 5
    use_levy_area: bool = False
    use_curvature: bool = True
    use_cumulative_sum: bool = False
    start_from_k3: bool = True
    max_dyadic_depth: int = 2  # Controls how many dyadic levels to use (2^0, 2^1, 2^2, ...)
    combine_dyadic_features: str = 'concatenate'  # 'concatenate', 'sum', 'mean'


@numba.njit
def compute_pairwise_volumes(ch_i: np.ndarray, ch_j: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Compute signed and unsigned pairwise volumes between two channels."""
    n = len(ch_i)
    signed_vols = np.zeros(n - 1)
    unsigned_vols = np.zeros(n - 1)

    for t in range(1, n):
        area = 0.5 * ((ch_i[t] - ch_i[t-1]) * (ch_j[t] + ch_j[t-1]) -
                      (ch_j[t] - ch_j[t-1]) * (ch_i[t] + ch_i[t-1]))
        signed_vols[t-1] = area
        unsigned_vols[t-1] = abs(area)

    return signed_vols, unsigned_vols


def compute_curvature_radius(data: np.ndarray) -> np.ndarray:
    """Compute curvature radius with padding to handle edge cases."""
    if len(data) < 3:
        return np.zeros(len(data))

    padded = np.concatenate([[data[0]], data, [data[-1]]])
    radius = np.zeros(len(data))

    for i in range(1, len(padded) - 1):
        denominator = abs(padded[i+1] - 2*padded[i] + padded[i-1]) + 0.001
        radius[i-1] = (1 + (padded[i+1] - padded[i])**2)**1.5 / denominator / 1000

    return radius


def extract_k1_features(channel: np.ndarray, config: Config) -> Tuple[List[float], List[float]]:
    """Extract K=1 features based on configuration."""
    signed_features, unsigned_features = [], []
    n = len(channel)

    if n < 2:
        return signed_features, unsigned_features

    if config.use_levy_area:
        chord_vals = np.linspace(channel[0], channel[-1], n)
        signed_vols, unsigned_vols = compute_pairwise_volumes(channel, chord_vals)
        signed_features.append(np.sum(signed_vols))
        unsigned_features.append(np.sum(unsigned_vols))

    if config.use_curvature:
        radius = compute_curvature_radius(channel)
        if len(radius) > 0:
            signed_vols, unsigned_vols = compute_pairwise_volumes(channel, radius)
            signed_features.append(np.sum(signed_vols))
            unsigned_features.append(np.sum(unsigned_vols))

    if config.use_cumulative_sum:
        signed_features.append(np.sum(channel))
        unsigned_features.append(np.sum(np.abs(channel)))

    return signed_features, unsigned_features


def get_dyadic_intervals(length: int, depth: int) -> List[Tuple[int, int]]:
    """Generate dyadic intervals for given depth.

    Args:
        length: Length of the time series
        depth: Dyadic depth (0 = whole series, 1 = 2 intervals, 2 = 4 intervals, etc.)

    Returns:
        List of (start, end) tuples for each dyadic interval
    """
    intervals = []
    n_intervals = 2 ** depth
    interval_length = length // n_intervals

    for i in range(n_intervals):
        start = i * interval_length
        end = start + interval_length if i < n_intervals - 1 else length
        intervals.append((start, end))

    return intervals


def extract_dyadic_features(X: List[np.ndarray], config: Config) -> Tuple[np.ndarray, np.ndarray]:
    """Extract features from dyadic intervals of the time series."""
    n_ch = X[0].shape[0]
    max_k = min(config.max_level, n_ch, 10)
    start_k = 3 if config.start_from_k3 else 2

    all_signed_features = []
    all_unsigned_features = []

    for x in X:
        n_samples = x.shape[1]
        sample_signed_by_depth = []
        sample_unsigned_by_depth = []

        # Process each dyadic depth
        for depth in range(config.max_dyadic_depth + 1):
            intervals = get_dyadic_intervals(n_samples, depth)
            depth_signed_features = []
            depth_unsigned_features = []

            for start_idx, end_idx in intervals:
                x_interval = x[:, start_idx:end_idx]

                if x_interval.shape[1] < 2:  # Skip intervals that are too small
                    continue

                interval_signed = []
                interval_unsigned = []

                # K=1 features for each channel in this interval
                for ch in range(n_ch):
                    ch_signed, ch_unsigned = extract_k1_features(x_interval[ch], config)
                    interval_signed.extend(ch_signed)
                    interval_unsigned.extend(ch_unsigned)

                # K=2 pairwise volumes for this interval
                if n_ch >= 2:
                    for ch_i, ch_j in combinations(range(n_ch), 2):
                        signed_vols, unsigned_vols = compute_pairwise_volumes(
                            x_interval[ch_i], x_interval[ch_j]
                        )
                        interval_signed.append(np.sum(signed_vols))
                        interval_unsigned.append(np.sum(unsigned_vols))

                # Higher-order combinations (K>=start_k) for this interval
                for k in range(start_k, max_k + 1):
                    if k > n_ch:
                        break
                    for combo in combinations(range(n_ch), k):
                        subset = x_interval[list(combo), :]
                        if subset.shape[1] > 0:
                            signed_density = np.prod(subset, axis=0)
                            unsigned_density = np.prod(np.abs(subset), axis=0)
                            interval_signed.append(np.sum(signed_density))
                            interval_unsigned.append(np.sum(unsigned_density))

                # Global feature for this interval (all channels)
                if n_ch <= max_k and n_ch >= start_k and x_interval.shape[1] > 0:
                    interval_signed.append(np.sum(np.prod(x_interval, axis=0)))
                    interval_unsigned.append(np.sum(np.prod(np.abs(x_interval), axis=0)))

                depth_signed_features.extend(interval_signed)
                depth_unsigned_features.extend(interval_unsigned)

            sample_signed_by_depth.append(depth_signed_features)
            sample_unsigned_by_depth.append(depth_unsigned_features)

        # Combine features across dyadic depths
        if config.combine_dyadic_features == 'concatenate':
            # Concatenate all features from all depths
            final_signed = []
            final_unsigned = []
            for depth_features in sample_signed_by_depth:
                final_signed.extend(depth_features)
            for depth_features in sample_unsigned_by_depth:
                final_unsigned.extend(depth_features)

        elif config.combine_dyadic_features == 'sum':
            # Sum corresponding features across depths
            max_len = max(len(depth_features) for depth_features in sample_signed_by_depth)
            final_signed = [0.0] * max_len
            final_unsigned = [0.0] * max_len

            for depth_features in sample_signed_by_depth:
                for i, feature in enumerate(depth_features):
                    if i < max_len:
                        final_signed[i] += feature

            for depth_features in sample_unsigned_by_depth:
                for i, feature in enumerate(depth_features):
                    if i < max_len:
                        final_unsigned[i] += feature

        elif config.combine_dyadic_features == 'mean':
            # Average corresponding features across depths
            max_len = max(len(depth_features) for depth_features in sample_signed_by_depth)
            final_signed = [0.0] * max_len
            final_unsigned = [0.0] * max_len
            depth_counts = [0] * max_len

            for depth_features in sample_signed_by_depth:
                for i, feature in enumerate(depth_features):
                    if i < max_len:
                        final_signed[i] += feature
                        depth_counts[i] += 1

            for depth_features in sample_unsigned_by_depth:
                for i, feature in enumerate(depth_features):
                    if i < max_len:
                        final_unsigned[i] += feature

            # Divide by counts to get mean
            for i in range(max_len):
                if depth_counts[i] > 0:
                    final_signed[i] /= depth_counts[i]
                    final_unsigned[i] /= depth_counts[i]

        all_signed_features.append(final_signed)
        all_unsigned_features.append(final_unsigned)

    # Convert to arrays, handle empty cases
    signed_arr = np.array(all_signed_features) if all_signed_features and all_signed_features[0] else np.empty((len(X), 0))
    unsigned_arr = np.array(all_unsigned_features) if all_unsigned_features and all_unsigned_features[0] else np.empty((len(X), 0))

    return signed_arr, unsigned_arr


def create_feature_sets(base_features: np.ndarray, signed_features: np.ndarray,
                       unsigned_features: np.ndarray) -> Dict[str, np.ndarray]:
    """Create all feature set combinations."""
    feature_sets = {'base': base_features}

    if signed_features.shape[1] > 0:
        feature_sets['signed'] = signed_features
        feature_sets['base_signed'] = np.concatenate([base_features, signed_features], axis=1)

    if unsigned_features.shape[1] > 0:
        feature_sets['unsigned'] = unsigned_features
        feature_sets['base_unsigned'] = np.concatenate([base_features, unsigned_features], axis=1)

    if signed_features.shape[1] > 0 and unsigned_features.shape[1] > 0:
        combined = np.concatenate([signed_features, unsigned_features], axis=1)
        feature_sets['signed_unsigned'] = combined
        feature_sets['base_signed_unsigned'] = np.concatenate([base_features, combined], axis=1)

    return feature_sets



def evaluate_with_cv(X: np.ndarray, y: np.ndarray, config: Config) -> Tuple[float, float, float, float]:
    """Evaluate features using k-fold cross validation WITHOUT any feature selection."""
    if X.shape[1] == 0:
        return 0.0, 0.0, 0.0, 0.0

    kfold = KFold(n_splits=config.k_folds, shuffle=True, random_state=config.random_state)
    acc_scores = []
    f1_scores = []

    for train_idx, val_idx in kfold.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # NO feature selection - use all features
        rf = RandomForestClassifier(n_estimators=config.n_estimators, random_state=config.random_state)
        rf.fit(X_train, y_train)
        pred = rf.predict(X_val)

        acc_scores.append(accuracy_score(y_val, pred))
        f1_scores.append(f1_score(y_val, pred, average='weighted'))

    return np.mean(acc_scores), np.std(acc_scores), np.mean(f1_scores), np.std(f1_scores)

def evaluate_all_features(feature_sets: Dict[str, np.ndarray], y: np.ndarray, config: Config) -> Dict[str, Dict]:
    """Evaluate all feature sets with cross-validation only (no feature selection)."""
    results = {}

    for name, X_features in feature_sets.items():
        acc_mean, acc_std, f1_mean, f1_std = evaluate_with_cv(X_features, y, config)
        results[name] = {
            'original': {
                'acc_mean': acc_mean,
                'acc_std': acc_std,
                'f1_mean': f1_mean,
                'f1_std': f1_std,
                'n_features': X_features.shape[1]
            }
        }

    return results

def print_results(results: Dict, dataset_name: str, k_folds: int):
    """Print evaluation results in a formatted table."""
    print(f"\n=== Results for {dataset_name} ({k_folds}-fold CV) ===")
    print(f"{'Feature Set':<25} {'Accuracy (mean±std)':<22} {'F1-Score (mean±std)':<22} {'# Features':<12}")
    print("-" * 85)

    display_order = ['base', 'signed', 'unsigned', 'signed_unsigned',
                    'base_signed', 'base_unsigned', 'base_signed_unsigned']

    for feature_name in display_order:
        if feature_name not in results:
            continue

        # Only show original results (no feature selection)
        metrics = results[feature_name]['original']
        acc_str = f"{metrics['acc_mean']:.4f}±{metrics['acc_std']:.4f}"
        f1_str = f"{metrics['f1_mean']:.4f}±{metrics['f1_std']:.4f}"
        print(f"{feature_name:<25} {acc_str:<22} {f1_str:<22} {metrics['n_features']:<12}")

    print()

def run_experiment(dataset_name: str, config: Config) -> Dict:
    """Run complete dyadic interval feature extraction and evaluation experiment."""
    print(f"Evaluating {dataset_name} with {config.k_folds}-fold cross validation (FIXED - no data leakage)...")
    print(f"Dyadic interval processing - Max depth: {config.max_dyadic_depth}, Combination: {config.combine_dyadic_features}")
    print(f"K=1 features enabled - Levy: {config.use_levy_area}, Curvature: {config.use_curvature}, CumSum: {config.use_cumulative_sum}")
    print(f"Higher-order combinations start from K={'3' if config.start_from_k3 else '2'}")

    # Load and process data
    X, y = load_classification(dataset_name)
    print(f"Number of channels: {X[0].shape[0]}")
    print(f"Number of classes: {len(np.unique(y))}")

    # Extract features (this is fine - no leakage here)
    base_features = np.array([iisignature.sig(x.T, config.sig_level) if x.size else np.empty(0) for x in X])
    signed_features, unsigned_features = extract_dyadic_features(X, config)
    feature_sets = create_feature_sets(base_features, signed_features, unsigned_features)

    # Print feature set sizes
    print("\nFeature set sizes:")
    for name, features in feature_sets.items():
        print(f"  {name}: {features.shape[1]} features")

    # Evaluate with fixed CV (no data leakage) and get average proportions
    results = evaluate_all_features(feature_sets, y, config)
    print_results(results, dataset_name, config.k_folds)

    return results

def compare_configurations(dataset_name: str, configs: List[Tuple[str, Config]]) -> Dict:
    """Compare multiple dyadic configurations side by side."""
    all_results = {}

    for config_name, config in configs:
        print(f"\n{'='*80}")
        print(f"Configuration: {config_name}")
        print(f"{'='*80}")

        results = run_experiment(dataset_name, config)
        all_results[config_name] = results

    return all_results

In [ ]:
import pandas as pd
from datetime import datetime

def run_multiple_classification_datasets():
    """Run experiments on multiple classification datasets and save results to CSVs with Accuracy and F1-Score."""
    # List of classification datasets to test
    datasets = [
    'ArticularyWordRecognition', 'AtrialFibrillation', 'BasicMotions', 'ERing',
    'HandMovementDirection', 'Handwriting',
    'Libras', 'LSST',
    'NATOPS',
    'PenDigits',
    'RacketSports', 'SelfRegulationSCP1', 'SelfRegulationSCP2',
    'StandWalkJump',
    'UWaveGestureLibrary'
    ]

    # Configuration to use
    config = Config(
        use_levy_area=False,
        use_curvature=True,
        use_cumulative_sum=False,
        # if we start from k2 its pretty bad
        start_from_k3=True
    )

    # Initialize results dictionary
    all_results = {}

    # Feature sets to track
    feature_sets = ['base', 'signed', 'unsigned', 'signed_unsigned',
                   'base_signed', 'base_unsigned', 'base_signed_unsigned']

    # Run experiments on each dataset
    for dataset in datasets:
        print(f"\n{'='*80}")
        print(f"Processing dataset: {dataset}")
        print(f"{'='*80}")

        try:
            # Run experiment
            results = run_experiment(dataset, config)

            # Create rows for this dataset
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}

            for feature_set in feature_sets:
                if feature_set in results:
                    # Original results
                    orig_metrics = results[feature_set]['original']
                    metrics_row[f'{feature_set}_original_accuracy'] = f"{orig_metrics['acc_mean']:.6f}±{orig_metrics['acc_std']:.6f}"
                    metrics_row[f'{feature_set}_original_f1'] = f"{orig_metrics['f1_mean']:.6f}±{orig_metrics['f1_std']:.6f}"
                    features_row[f'{feature_set}_original'] = orig_metrics['n_features_original']

                else:
                    # Fill with NaN if feature set not available
                    metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                    metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                    features_row[f'{feature_set}_original'] = 0

            all_results[dataset] = (metrics_row, features_row)

        except Exception as e:
            print(f"Error processing {dataset}: {str(e)}")
            # Add error rows for failed datasets
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}
            for feature_set in feature_sets:
                metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                features_row[f'{feature_set}_original'] = 0
            all_results[dataset] = (metrics_row, features_row)
            continue

    # Separate metrics and features data
    metrics_data = {}
    features_data = {}

    for dataset, (metrics_row, features_row) in all_results.items():
        metrics_data[dataset] = metrics_row
        features_data[dataset] = features_row

    # Convert to DataFrames
    metrics_df = pd.DataFrame.from_dict(metrics_data, orient='index')
    features_df = pd.DataFrame.from_dict(features_data, orient='index')

    # Generate timestamp for unique filenames
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save to CSV files
    metrics_filename = f'classification_metrics_results_{timestamp}.csv'
    features_filename = f'classification_feature_counts_{timestamp}.csv'

    metrics_df.to_csv(metrics_filename, index=False)
    features_df.to_csv(features_filename, index=False)

    print(f"\n{'='*80}")
    print(f"Results saved to:")
    print(f"Metrics (Accuracy/F1): {metrics_filename}")
    print(f"Feature Counts: {features_filename}")
    print(f"{'='*80}")

    # Display summary
    successful_datasets = metrics_df[metrics_df['dataset'].notna()].shape[0]
    print(f"\nSummary:")
    print(f"Total datasets attempted: {len(datasets)}")
    print(f"Successfully processed: {successful_datasets}")

    # Show sample of results
    print(f"\nFirst few rows of metrics:")
    print(metrics_df.head())

    print(f"\nFirst few rows of feature counts:")
    print(features_df.head())

    return metrics_df, features_df

# Run the experiments
metrics_df, features_df = run_multiple_classification_datasets()

In [ ]:
import pandas as pd
from datetime import datetime

def run_multiple_classification_datasets():
    """Run experiments on multiple classification datasets and save results to CSVs with Accuracy and F1-Score."""
    # List of classification datasets to test
    datasets = [
    'ArticularyWordRecognition', 'AtrialFibrillation', 'BasicMotions', 'ERing',
    'HandMovementDirection', 'Handwriting',
    'Libras', 'LSST',
    'NATOPS',
    'PenDigits',
    'RacketSports', 'SelfRegulationSCP1', 'SelfRegulationSCP2',
    'StandWalkJump',
    'UWaveGestureLibrary'
    ]


    # Configuration to use
    config = Config(
        use_levy_area=True,
        use_curvature=True,
        use_cumulative_sum=False,
        # if we start from k2 its pretty bad
        start_from_k3=True
    )

    # Initialize results dictionary
    all_results = {}

    # Feature sets to track
    feature_sets = ['base', 'signed', 'unsigned', 'signed_unsigned',
                   'base_signed', 'base_unsigned', 'base_signed_unsigned']

    # Run experiments on each dataset
    for dataset in datasets:
        print(f"\n{'='*80}")
        print(f"Processing dataset: {dataset}")
        print(f"{'='*80}")

        try:
            # Run experiment
            results = run_experiment(dataset, config)

            # Create rows for this dataset
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}

            for feature_set in feature_sets:
                if feature_set in results:
                    # Original results
                    orig_metrics = results[feature_set]['original']
                    metrics_row[f'{feature_set}_original_accuracy'] = f"{orig_metrics['acc_mean']:.6f}±{orig_metrics['acc_std']:.6f}"
                    metrics_row[f'{feature_set}_original_f1'] = f"{orig_metrics['f1_mean']:.6f}±{orig_metrics['f1_std']:.6f}"
                    features_row[f'{feature_set}_original'] = orig_metrics['n_features_original']

                else:
                    # Fill with NaN if feature set not available
                    metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                    metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                    features_row[f'{feature_set}_original'] = 0

            all_results[dataset] = (metrics_row, features_row)

        except Exception as e:
            print(f"Error processing {dataset}: {str(e)}")
            # Add error rows for failed datasets
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}
            for feature_set in feature_sets:
                metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                features_row[f'{feature_set}_original'] = 0
            all_results[dataset] = (metrics_row, features_row)
            continue

    # Separate metrics and features data
    metrics_data = {}
    features_data = {}

    for dataset, (metrics_row, features_row) in all_results.items():
        metrics_data[dataset] = metrics_row
        features_data[dataset] = features_row

    # Convert to DataFrames
    metrics_df = pd.DataFrame.from_dict(metrics_data, orient='index')
    features_df = pd.DataFrame.from_dict(features_data, orient='index')

    # Generate timestamp for unique filenames
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save to CSV files
    metrics_filename = f'classification_metrics_results_{timestamp}.csv'
    features_filename = f'classification_feature_counts_{timestamp}.csv'

    metrics_df.to_csv(metrics_filename, index=False)
    features_df.to_csv(features_filename, index=False)

    print(f"\n{'='*80}")
    print(f"Results saved to:")
    print(f"Metrics (Accuracy/F1): {metrics_filename}")
    print(f"Feature Counts: {features_filename}")
    print(f"{'='*80}")

    # Display summary
    successful_datasets = metrics_df[metrics_df['dataset'].notna()].shape[0]
    print(f"\nSummary:")
    print(f"Total datasets attempted: {len(datasets)}")
    print(f"Successfully processed: {successful_datasets}")

    # Show sample of results
    print(f"\nFirst few rows of metrics:")
    print(metrics_df.head())

    print(f"\nFirst few rows of feature counts:")
    print(features_df.head())

    return metrics_df, features_df

# Run the experiments
metrics_df, features_df = run_multiple_classification_datasets()

In [ ]:
import pandas as pd
from datetime import datetime

def run_multiple_classification_datasets():
    """Run experiments on multiple classification datasets and save results to CSVs with Accuracy and F1-Score."""
    # List of classification datasets to test
    datasets = datasets = [
    "AtrialFibrillation",
    "HandMovementDirection",
    "Handwriting",
    "Libras",
    "LSST",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump"
    ]


    # Configuration to use
    config = Config(
        use_levy_area=True,
        use_curvature=True,
        use_cumulative_sum=True,
        # if we start from k2 its pretty bad
        start_from_k3=True
    )

    # Initialize results dictionary
    all_results = {}

    # Feature sets to track
    feature_sets = ['base', 'signed', 'unsigned', 'signed_unsigned',
                   'base_signed', 'base_unsigned', 'base_signed_unsigned']

    # Run experiments on each dataset
    for dataset in datasets:
        print(f"\n{'='*80}")
        print(f"Processing dataset: {dataset}")
        print(f"{'='*80}")

        try:
            # Run experiment
            results = run_experiment(dataset, config)

            # Create rows for this dataset
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}

            for feature_set in feature_sets:
                if feature_set in results:
                    # Original results
                    orig_metrics = results[feature_set]['original']
                    metrics_row[f'{feature_set}_original_accuracy'] = f"{orig_metrics['acc_mean']:.6f}±{orig_metrics['acc_std']:.6f}"
                    metrics_row[f'{feature_set}_original_f1'] = f"{orig_metrics['f1_mean']:.6f}±{orig_metrics['f1_std']:.6f}"
                    features_row[f'{feature_set}_original'] = orig_metrics['n_features']

                else:
                    # Fill with NaN if feature set not available
                    metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                    metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                    features_row[f'{feature_set}_original'] = 0

            all_results[dataset] = (metrics_row, features_row)

        except Exception as e:
            print(f"Error processing {dataset}: {str(e)}")
            # Add error rows for failed datasets
            metrics_row = {'dataset': dataset}
            features_row = {'dataset': dataset}
            for feature_set in feature_sets:
                metrics_row[f'{feature_set}_original_accuracy'] = 'NaN'
                metrics_row[f'{feature_set}_original_f1'] = 'NaN'
                features_row[f'{feature_set}_original'] = 0
            all_results[dataset] = (metrics_row, features_row)
            continue

    # Separate metrics and features data
    metrics_data = {}
    features_data = {}

    for dataset, (metrics_row, features_row) in all_results.items():
        metrics_data[dataset] = metrics_row
        features_data[dataset] = features_row

    # Convert to DataFrames
    metrics_df = pd.DataFrame.from_dict(metrics_data, orient='index')
    features_df = pd.DataFrame.from_dict(features_data, orient='index')

    # Generate timestamp for unique filenames
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save to CSV files
    metrics_filename = f'classification_metrics_results_{timestamp}.csv'
    features_filename = f'classification_feature_counts_{timestamp}.csv'

    metrics_df.to_csv(metrics_filename, index=False)
    features_df.to_csv(features_filename, index=False)

    print(f"\n{'='*80}")
    print(f"Results saved to:")
    print(f"Metrics (Accuracy/F1): {metrics_filename}")
    print(f"Feature Counts: {features_filename}")
    print(f"{'='*80}")

    # Display summary
    successful_datasets = metrics_df[metrics_df['dataset'].notna()].shape[0]
    print(f"\nSummary:")
    print(f"Total datasets attempted: {len(datasets)}")
    print(f"Successfully processed: {successful_datasets}")

    # Show sample of results
    print(f"\nFirst few rows of metrics:")
    print(metrics_df.head())

    print(f"\nFirst few rows of feature counts:")
    print(features_df.head())

    return metrics_df, features_df

# Run the experiments
metrics_df, features_df = run_multiple_classification_datasets()

In [ ]:
import zipfile
import glob
from google.colab import files

# Match the target files
mse_files = glob.glob('*metrics_results_*.csv')
feature_files = glob.glob('*feature_counts_*.csv')

all_files = mse_files + feature_files
zip_filename = "colab_resultsmc.zip"

# Create a ZIP file containing all matched files
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in all_files:
        print(f"Adding {file} to ZIP")
        zipf.write(file)

# Download the ZIP file
files.download(zip_filename)

print("✅ All files zipped and downloaded as colab_results.zip!")
